# 03 · Fase 3 — Memória: o agente vira chatbot

Rubricas **3.1 a 3.6**: taxonomia CoALA, gestão da janela de contexto (sliding window, sumarização e busca vetorial), comparação de ferramentas de memória, chatbot com memória de curto prazo, NLU com intenção e entidades, e conversa multi-turno real.

> **Sobre o modo offline.** O enunciado pede alternativas *dummy* para APIs e credenciais. Sem `OPENAI_API_KEY` (ou com `MODO_OFFLINE=true`), o projeto troca o GPT por um **modelo roteirizado** (`src/llm.py`: regras determinísticas que escolhem a ferramenta e formatam a observação), os embeddings por `DeterministicFakeEmbedding` e as APIs por **fixtures gravados** de chamadas reais (`fixtures/`). A estrutura das chains, do agente, da memória e da verificação é exatamente a mesma; só a "inteligência" muda. As saídas deste notebook foram geradas no modo indicado na célula de configuração.

## 3.1 Curto e longo prazo com a taxonomia CoALA

| Camada CoALA | No ApuraCota | Implementação |
|---|---|---|
| **STM / working memory** | turnos da apuração em andamento | `MemorySaver` (checkpointer) por `thread_id`; poda com `trim_messages` |
| **Cache semântico** | respostas de API já obtidas nesta sessão | `_get_json` em `src/tools.py` grava e reaproveita por URL+parâmetros (vira os fixtures do modo offline) |
| **LTM semântica** | regras da cota (o que pode, limites, prazos) | FAISS do Ato da Mesa 43/2009 (`consultar_regras_cota`) |
| **LTM episódica** | apurações anteriores, com data e repórter | `InMemoryVectorStore` + ferramentas `gravar_apuracao` / `lembrar_apuracao` |
| **LTM procedural** | como agir (rotas, formato, "nunca inventar") | system prompt PTCF e docstrings das ferramentas |

In [1]:
# Configuração comum: roda a partir da raiz do repositório e mostra o modo de execução
import os
import sys
from importlib.metadata import version
from pathlib import Path

RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(RAIZ)
sys.path.insert(0, str(RAIZ))
import warnings

warnings.filterwarnings("ignore")

from src import config, llm

print(
    "LangChain",
    version("langchain"),
    "| langchain-core",
    version("langchain-core"),
    "| LangGraph",
    version("langgraph"),
    "| LangSmith",
    version("langsmith"),
)
print(
    "Modo:",
    "OFFLINE (modelo dummy roteirizado + fixtures)" if llm.offline() else f"ONLINE ({config.MODELO_AGENTE})",
)

LangChain 1.4.9 | langchain-core 1.7.6 | LangGraph 1.2.20 | LangSmith 0.14.17
Modo: OFFLINE (modelo dummy roteirizado + fixtures)


## 3.2 Gestão da janela de contexto

Três mecanismos combinados: (1) **avaliador de relevância** marca mensagens de conversa fiada como descartáveis; (2) **sliding window por tokens** (`trim_messages`) mantém só o que cabe; (3) **sumarização** condensa a apuração e grava o resumo na memória episódica, de onde a **busca vetorial** recupera quando o repórter voltar ao assunto.

In [2]:
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate

prompt_relevancia = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "Classifique a relevância da mensagem para uma apuração de gastos parlamentares. Responda apenas 'alta' ou 'baixa'.",
        ),
        ("human", "{mensagem}"),
    ]
)
avaliar_relevancia = prompt_relevancia | llm.chat_model() | StrOutputParser()

mensagens = [
    "Quanto a Tabata Amaral gastou com combustível em 2025?",
    "haha, que calor hoje em São Paulo",
    "O fornecedor Posto Batalha recebeu R$ 17.552,73 dela.",
    "me lembra de almoçar",
]
for m in mensagens:
    print(f"{avaliar_relevancia.invoke({'mensagem': m}).strip():6} <- {m}")

alta   <- Quanto a Tabata Amaral gastou com combustível em 2025?
baixa  <- haha, que calor hoje em São Paulo
alta   <- O fornecedor Posto Batalha recebeu R$ 17.552,73 dela.
baixa  <- me lembra de almoçar


In [3]:
from src.memory import podar_historico

historico = [SystemMessage("ApuraCota")] + [
    x
    for i in range(8)
    for x in (HumanMessage(f"pergunta {i} " + "." * 300), AIMessage(f"resposta {i} " + "." * 300))
]
podado = podar_historico(historico, llm.chat_model(), max_tokens=600)
print(
    f"sliding window: {len(historico)} -> {len(podado)} mensagens (system preservado: {podado[0].type == 'system'})"
)

sliding window: 17 -> 7 mensagens (system preservado: True)


In [4]:
from src.memory import SYSTEM_RESUMO, gravar_apuracao, lembrar_apuracao

conversa = [
    HumanMessage("Quanto a Tabata Amaral gastou com combustível em 2025?"),
    AIMessage("Tabata Amaral (PSB-SP) gastou R$ 48.808,83 com combustíveis em 2025. Fonte: API da Câmara."),
    HumanMessage("E a média do partido?"),
    AIMessage("A média por deputado do PSB foi R$ 48.849,91, praticamente igual (-0,08%)."),
]
prompt_resumo = ChatPromptTemplate.from_messages([SYSTEM_RESUMO, ("human", "{conversa}")])
resumo = (prompt_resumo | llm.chat_model() | StrOutputParser()).invoke(
    {"conversa": "\n".join(f"{m.type}: {m.content}" for m in conversa)}
)
print("RESUMO:", resumo, "\n")
print(gravar_apuracao.invoke({"resumo": resumo, "reporter_id": "igor"}))
print(
    gravar_apuracao.invoke(
        {
            "resumo": "Nikolas Ferreira gastou R$ 150.349,17 com locação de veículos em 2025.",
            "reporter_id": "igor",
        }
    )
)
print(
    "\nBUSCA VETORIAL:", lembrar_apuracao.invoke({"consulta": "combustível da Tabata", "reporter_id": "igor"})
)
print("\nOUTRO REPÓRTER:", lembrar_apuracao.invoke({"consulta": "combustível", "reporter_id": "ana"}))

RESUMO: Resumo (offline): human: Quanto a Tabata Amaral gastou com combustível em 2025? ai: Tabata Amaral (PSB-SP) gastou R$ 48.808,83 com combustíveis em 2025. Fonte: API da Câmara. human: E a média do partido? ai: A média por deputado do PSB foi R$ 48.849,91, praticamente igual (-0,08%). 

Apuracao salva para 'igor' em 2026-11-07.
Apuracao salva para 'igor' em 2026-11-07.

BUSCA VETORIAL: * [2026-11-07] Nikolas Ferreira gastou R$ 150.349,17 com locação de veículos em 2025.
* [2026-11-07] Resumo (offline): human: Quanto a Tabata Amaral gastou com combustível em 2025? ai: Tabata Amaral (PSB-SP) gastou R$ 48.808,83 com combustíveis em 2025. Fonte: API da Câmara. human: E a média do partido? ai: A média por deputado do PSB foi R$ 48.849,91, praticamente igual (-0,08%).

OUTRO REPÓRTER: Nenhuma apuracao anterior encontrada.


## 3.3 Ferramentas especializadas de memória

| Opção | O que oferece | Custo/risco | Veredito para o ApuraCota |
|---|---|---|---|
| **Custom LangChain/LangGraph** (checkpointer + vector store) | controle total, sem serviço externo, grátis | implementar decaimento e consolidação à mão | **escolhida**: volume pequeno, dados de apuração não devem sair da redação |
| Mem0 | extração automática de fatos, memória por usuário, API hospedada | dados em terceiro; custo por chamada | descartada por governança dos dados |
| Letta (MemGPT) | agente com memória hierárquica autogerida | serviço/infra própria, curva de aprendizado | excesso para um agente único |
| LangMem | consolidação de memória em LangGraph | ainda jovem, mais útil com LangGraph Store | candidata para a próxima disciplina |

## 3.4 Chatbot com memória de curto prazo

> **Nota de equivalência (rubrica 3.4).** `ConversationBufferMemory` pertence ao módulo `langchain.memory`, legado. O buffer de curto prazo é o **checkpointer `MemorySaver`** indexado por `thread_id`: cada apuração tem a sua thread, e threads são isoladas.

In [5]:
from src.agent import montar_agente, trajetoria

bot = montar_agente()
t1 = {"configurable": {"thread_id": "apuracao-A"}}
t2 = {"configurable": {"thread_id": "apuracao-B"}}
bot.invoke({"messages": [("user", "Quanto a Tabata Amaral gastou com combustível em 2025?")]}, config=t1)
print(
    "thread A, turno 2:",
    bot.invoke({"messages": [("user", "Compare com a média do partido dela.")]}, config=t1)["messages"][
        -1
    ].content[:160],
)
print(
    "thread B, sem contexto:",
    bot.invoke({"messages": [("user", "Compare com a média do partido dela.")]}, config=t2)["messages"][
        -1
    ].content[:160],
)

[AUDITORIA SQL] SELECT ROUND(AVG(t.total),2) AS media_por_deputado FROM (SELECT id_deputado, SUM(valor) AS total FROM despesas_ceap WHERE partido='PSB' AND ano=2025 AND categoria LIKE '%COMBUST%' GROUP BY id_deputado) t LIMIT 200
thread A, turno 2: Media por deputado: R$ 48.849,91 em 2025. Fonte: CSV CEAP 2025 (banco local).
thread B, sem contexto: Para comparar preciso saber o partido, a UF ou o deputado de referencia.


## 3.5 Reconhecimento de intenção e extração de entidades (NLU)

`AnaliseNLU` é preenchida por `with_structured_output` a partir do histórico (`MessagesPlaceholder`) e da mensagem atual. O campo `herdado_do_historico` mostra o que veio de turnos anteriores.

In [6]:
from src.chains import analisar_turno

hist = [
    HumanMessage("Quanto a Tabata Amaral gastou com combustível em 2025?"),
    AIMessage("R$ 48.808,83. Fonte: API da Câmara."),
]
for msg in ["E com passagens aéreas?", "Qual fornecedor de combustível recebeu mais dela?", "Obrigado!"]:
    a = analisar_turno(hist, msg)
    print(f"{msg!r}\n   -> {a.model_dump(exclude_none=True)}")

'E com passagens aéreas?'
   -> {'intencao': 'consultar_gasto', 'deputado': 'API Câmara', 'periodo': '2025', 'categoria': 'PASSAGEM', 'herdado_do_historico': ['deputado', 'periodo']}
'Qual fornecedor de combustível recebeu mais dela?'
   -> {'intencao': 'ranking_fornecedor', 'deputado': 'API Câmara', 'periodo': '2025', 'categoria': 'COMBUST', 'herdado_do_historico': ['deputado', 'periodo']}
'Obrigado!'
   -> {'intencao': 'consultar_gasto', 'deputado': 'Obrigado', 'periodo': '2025', 'categoria': 'COMBUST', 'herdado_do_historico': ['periodo', 'categoria']}


## 3.6 Conversa multi-turno real (contexto mantido e decomposição)

In [7]:
roteiro = [
    "Quanto o deputado Nikolas Ferreira gastou com combustível em 2025?",
    "E com passagens aéreas?",
    "Compare esse gasto com a média do PL.",
    "Quais os maiores fornecedores dele?",
    "Quem é o fornecedor de CNPJ 33.927.528/0001-60?",
]
chat = montar_agente()
cfg = {"configurable": {"thread_id": "conversa-3.6"}}
for i, pergunta in enumerate(roteiro, 1):
    est = chat.invoke({"messages": [("user", pergunta)]}, config=cfg)
    usadas = [(p["ferramenta"], p["args"]) for p in trajetoria(est) if p["tipo"] == "decisao"]
    print(f"\n[{i}] USUÁRIO: {pergunta}")
    print(f"    FERRAMENTAS: {usadas}")
    print(f"    AGENTE: {est['messages'][-1].content[:260]}")


[1] USUÁRIO: Quanto o deputado Nikolas Ferreira gastou com combustível em 2025?
    FERRAMENTAS: [('buscar_deputado', {'nome': 'Nikolas Ferreira', 'uf': None}), ('listar_despesas', {'id_deputado': 209787, 'ano': 2025, 'categoria': 'COMBUST'})]
    AGENTE: Nikolas Ferreira gastou R$ 17.190,70 com combust em 2025 (91 lancamentos). Maiores categorias: Combustíveis E Lubrificantes.: R$ 17.190,70. Fonte: API da Camara, despesas de 2025.



[2] USUÁRIO: E com passagens aéreas?
    FERRAMENTAS: [('listar_despesas', {'id_deputado': 209787, 'ano': 2025, 'categoria': 'PASSAGEM'})]
    AGENTE: Nikolas Ferreira gastou R$ 158.590,29 com passagem em 2025 (221 lancamentos). Maiores categorias: Passagem Aérea - Sigepa: R$ 158.590,29. Fonte: API da Camara, despesas de 2025.
[AUDITORIA SQL] SELECT ROUND(AVG(t.total),2) AS media_por_deputado FROM (SELECT id_deputado, SUM(valor) AS total FROM despesas_ceap WHERE partido='PL' AND ano=2025 AND categoria LIKE '%PASSAGEM%' GROUP BY id_deputado) t LIMIT 200



[3] USUÁRIO: Compare esse gasto com a média do PL.
    FERRAMENTAS: [('consultar_ceap', {'query_sql': "SELECT ROUND(AVG(t.total),2) AS media_por_deputado FROM (SELECT id_deputado, SUM(valor) AS total FROM despesas_ceap WHERE partido='PL' AND ano=2025 AND categoria LIKE '%PASSAGEM%' GROUP BY id_deputado) t"})]
    AGENTE: Media por deputado: R$ 56.788,70 em 2025. Fonte: CSV CEAP 2025 (banco local).
[AUDITORIA SQL] SELECT fornecedor, cnpj_cpf, ROUND(SUM(valor),2) AS total FROM despesas_ceap WHERE id_deputado=209787 AND ano=2025 AND categoria LIKE '%PASSAGEM%' GROUP BY 1,2 ORDER BY total DESC LIMIT 5



[4] USUÁRIO: Quais os maiores fornecedores dele?
    FERRAMENTAS: [('consultar_ceap', {'query_sql': "SELECT fornecedor, cnpj_cpf, ROUND(SUM(valor),2) AS total FROM despesas_ceap WHERE id_deputado=209787 AND ano=2025 AND categoria LIKE '%PASSAGEM%' GROUP BY 1,2 ORDER BY total DESC LIMIT 5"})]
    AGENTE: Resultado do banco local da CEAP:
- TAM | 02012862000160: R$ 42.267,48
- AZUL | 09296295000160: R$ 24.420,25
- GOL | 07575651000159: R$ 17.731,12
Fonte: CSV CEAP 2025 (banco local).

[5] USUÁRIO: Quem é o fornecedor de CNPJ 33.927.528/0001-60?
    FERRAMENTAS: [('dados_fornecedor', {'cnpj': '33.927.528/0001-60'})]
    AGENTE: Dados do fornecedor (Brasil API): {"razao_social": "SJR IMOVEIS LTDA", "situacao": "ATIVA", "cnae_principal": "Corretagem na compra e venda e avaliação de imóveis", "municipio": "SAO PAULO-SP", "abertura": "2019-06-14"}


**Leitura da conversa.** O turno 2 herda deputado e ano; o turno 3 herda a categoria e descobre o partido no contexto para montar a média; o turno 4 usa o id resolvido no turno 1 para o ranking em SQL. Nenhum dado foi repetido pelo usuário. Limitação a registrar no relatório: a memória episódica está em memória RAM (reinicia com o processo); persistir em FAISS em disco é a melhoria natural.